In [1]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN").strip()
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    login(token=hf_token, add_to_git_credential=False)
    print("[AUTH] Successfully authenticated with Hugging Face Hub.")
except Exception as e:
    print(f"[AUTH WARNING] Could not load HF_TOKEN from Kaggle Secrets: {e}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[AUTH] Successfully authenticated with Hugging Face Hub.


In [2]:
import ast, hashlib, inspect, json, os, shutil, sys, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
import torch

# Optional: authenticate HF_TOKEN if attached in Secrets
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    hf_token = UserSecretsClient().get_secret("HF_TOKEN").strip()
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token, add_to_git_credential=False)
except Exception:
    pass

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/strict_audit_workspace")
WORK_DIR.mkdir(parents=True, exist_ok=True)

zip_path = next(INPUT_ROOT.rglob("kaggle_working_full_backup.zip"))
EXTRACT_DIR = WORK_DIR / "extracted"
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(EXTRACT_DIR)

HARNESS_DIR = next(EXTRACT_DIR.rglob("kv_spa_harness")).parent
RUN_DIR = next(EXTRACT_DIR.rglob("ten_model_production_run"))
sys.path.insert(0, str(HARNESS_DIR))

from kv_spa_harness.dataset import PERSONA_REGISTRY
from kv_spa_harness.metrics_and_stats import (
    LocalSemanticPreservationEvaluator,
    compute_benjamini_hochberg_fdr,
    compute_cohens_d_with_bootstrap_ci,
    compute_mattr,
    compute_target_marker_density,
    count_hard_constraint_violations,
)
import kv_spa_harness.baselines as m_base
import kv_spa_harness.belitsky_steering as m_bel
import kv_spa_harness.sun_transplantation as m_sun
import kv_spa_harness.memory_inception as m_mem
import kv_spa_harness.runner as m_run

# ==============================================================================
# STAGE 1: BYTE-LEVEL CONTENT INTEGRITY & EXECUTION COMPLETENESS
# ==============================================================================
zip_sha256 = hashlib.sha256(zip_path.read_bytes()).hexdigest()
json_files = sorted(
    p for p in RUN_DIR.rglob("*.json")
    if p.stem.startswith(("A1_", "A2_", "A3_", "A4_", "A5_", "A6_"))
)

content_hasher = hashlib.sha256()
records = []
for p in json_files:
    raw_bytes = p.read_bytes()
    content_hasher.update(p.name.encode("utf-8") + b"\x00" + hashlib.sha256(raw_bytes).digest())
    records.append(json.loads(raw_bytes.decode("utf-8")))

corpus_content_sha256 = content_hasher.hexdigest()

# Verify 6 arms x 10 models x 4 personas = 240 unique cells, 50 turns each
grid_cells = {(r["arm_id"], r["ollama_model_id"], r["persona_id"]) for r in records}
turn_counts = [len(r.get("turn_records", [])) for r in records]
assert len(json_files) == 240 and len(grid_cells) == 240 and all(c == 50 for c in turn_counts)

# Separate audit of 0-token turns vs. repetition collapse (independent phenomena)
empty_turns = []
collapse_with_empty = 0
for r in records:
    emp_t = [t["turn_index"] for t in r["turn_records"] if not str(t.get("raw_model_response", "")).strip()]
    if emp_t:
        empty_turns.extend([(r["trajectory_id"], r["ollama_model_id"], r["arm_id"], ti) for ti in emp_t])
        # Check if MATTR collapse flag (T46-50 vs T1-5) overlapped with any empty turn in T1-5 or T46-50
        boundary_emp = [ti for ti in emp_t if ti <= 5 or ti >= 46]
        if boundary_emp:
            collapse_with_empty += 1

print("=" * 88)
print("STAGE 1: BYTE-LEVEL DATA INTEGRITY & EXECUTION COMPLETENESS")
print("=" * 88)
print(f"  Archive Byte SHA-256 : {zip_sha256}")
print(f"  240-JSON Content SHA : {corpus_content_sha256}")
print(f"  Factorial Grid       : {len(grid_cells)}/240 unique (arm, model, persona) trajectories x 50 turns = 12,000 turns")
print(f"  Empty Response Audit : {len(empty_turns)}/12,000 turns (0.19%) across {len({x[0] for x in empty_turns})} trajectories")
print(f"  Boundary Overlap     : {collapse_with_empty} trajectories had empty turns inside T1-5 or T46-50 MATTR windows")

# ==============================================================================
# STAGE 2: IMPLEMENTATION AUDIT (SOURCE AST + JSON TELEMETRY + HOOK VERIFICATION)
# ==============================================================================
print("\n" + "=" * 88)
print("STAGE 2: ARM IMPLEMENTATION & MECHANISM VERIFICATION (A1-A6)")
print("=" * 88)

for mod_name, mod_obj in [
    ("baselines (A1/A2/A3)", m_base),
    ("belitsky_steering (A4)", m_bel),
    ("sun_transplantation (A5)", m_sun),
    ("memory_inception (A6)", m_mem),
    ("runner (Dispatcher)", m_run),
]:
    src = inspect.getsource(mod_obj)
    tree = ast.parse(src)
    funcs = [n.name for n in ast.walk(tree) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]
    classes = [n.name for n in ast.walk(tree) if isinstance(n, ast.ClassDef)]
    mod_sha = hashlib.sha256(src.encode("utf-8")).hexdigest()[:12]
    print(f"  Module {mod_name:<26} | SHA256[:12]={mod_sha} | Classes={classes} | Funcs={funcs}")

# Inspect trajectory-level & turn-level metadata keys per arm to verify distinct execution paths
for arm in ["A1", "A2", "A3", "A4", "A5", "A6"]:
    arm_recs = [r for r in records if r["arm_id"] == arm]
    sample_r = arm_recs[0]
    sample_t1 = sample_r["turn_records"][0]
    top_keys = sorted(set(sample_r.keys()) - {"turn_records", "trajectory_id", "arm_id", "persona_id", "ollama_model_id", "hf_checkpoint", "scenario_seed", "total_turns_completed", "total_execution_time_sec"})
    turn_keys = sorted(set(sample_t1.keys()) - {"turn_index", "raw_model_response", "time_to_first_token_ms", "inter_token_latency_ms", "total_turn_latency_ms", "peak_vram_allocated_mb", "input_prompt_token_count", "generated_token_count"})
    t1_toks = np.mean([r["turn_records"][0]["input_prompt_token_count"] for r in arm_recs])
    t50_toks = np.mean([r["turn_records"][49]["input_prompt_token_count"] for r in arm_recs])
    print(f"  Arm {arm}: T1_toks={t1_toks:.1f} -> T50_toks={t50_toks:.1f} | Extra Traj Keys={top_keys} | Extra Turn Keys={turn_keys}")

# Print exact runner dispatch logic for A1-A6 from runner.py so reviewers can read the exact code path
print("\n  --- Runner Arm Dispatch Source Excerpt (runner.py) ---")
runner_src_lines = inspect.getsource(m_run).splitlines()
for i, line in enumerate(runner_src_lines, 1):
    if any(k in line for k in ("A1", "A2", "A3", "A4", "A5", "A6", "belitsky", "sun", "inception", "spa", "reminder")):
        print(f"    L{i:03d}: {line}")

# ==============================================================================
# STAGE 3: FULL 12,000-TURN & 240-TRAJECTORY BITWISE METRIC REPRODUCTION
# ==============================================================================
print("\n" + "=" * 88)
print("STAGE 3: FULL 12,000-TURN METRIC RECOMPUTATION & CSV CROSS-CHECK")
print("=" * 88)

evaluator = LocalSemanticPreservationEvaluator()
re_turn_rows, re_traj_rows = [], []

for r in records:
    traj_id = r["trajectory_id"]
    persona = PERSONA_REGISTRY[r["persona_id"]]
    p_list, m_list, d_list, v_list = [], [], [], []

    for t in r["turn_records"]:
        resp = t["raw_model_response"]
        p_sem = evaluator.compute_p_sem(resp, persona)
        mattr = compute_mattr(resp, window_size=50)
        dens = compute_target_marker_density(resp, persona.diagnostic_lexical_markers)
        viol = count_hard_constraint_violations(resp, persona.forbidden_tokens_or_phrases)

        p_list.append(p_sem)
        m_list.append(mattr)
        d_list.append(dens)
        v_list.append(viol)
        re_turn_rows.append({
            "trajectory_id": traj_id,
            "turn_index": int(t["turn_index"]),
            "p_sem": p_sem,
            "mattr_50": mattr,
            "marker_density": dens,
            "hard_violations": viol,
        })

    re_traj_rows.append({
        "trajectory_id": traj_id,
        "arm_id": r["arm_id"],
        "model_id": r["ollama_model_id"],
        "persona_id": r["persona_id"],
        "cell_id": f"{r['ollama_model_id']}__{r['persona_id']}",
        "mean_p_sem": float(np.mean(p_list)),
        "mean_mattr": float(np.mean(m_list)),
        "delta_mattr_vs_init": float(np.mean(m_list[-5:]) - np.mean(m_list[:5])),
        "repetition_collapse_flag": bool((np.mean(m_list[-5:]) - np.mean(m_list[:5])) < -0.08),
        "mean_marker_density": float(np.mean(d_list)),
        "total_hard_violations": int(np.sum(v_list)),
    })

re_turns_df = pd.DataFrame(re_turn_rows).sort_values(["trajectory_id", "turn_index"]).reset_index(drop=True)
re_traj_df = pd.DataFrame(re_traj_rows).sort_values("trajectory_id").reset_index(drop=True)

# Locate saved CSVs if attached in /kaggle/input or /kaggle/working
csv_turns_matches = list(INPUT_ROOT.rglob("full_12000_turn_level_telemetry.csv")) + list(Path("/kaggle/working").rglob("full_12000_turn_level_telemetry.csv"))
csv_traj_matches = list(INPUT_ROOT.rglob("full_240_trajectory_summary_with_models.csv")) + list(Path("/kaggle/working").rglob("full_240_trajectory_summary_with_models.csv"))

if csv_turns_matches and csv_traj_matches:
    orig_turns = pd.read_csv(csv_turns_matches[0]).sort_values(["trajectory_id", "turn_index"]).reset_index(drop=True)
    orig_traj = pd.read_csv(csv_traj_matches[0]).sort_values("trajectory_id").reset_index(drop=True)

    max_psem_err = float(np.max(np.abs(re_turns_df["p_sem"] - orig_turns["p_sem"])))
    max_mattr_err = float(np.max(np.abs(re_turns_df["mattr_50"] - orig_turns["mattr_50"])))
    max_dens_err = float(np.max(np.abs(re_turns_df["marker_density"] - orig_turns["marker_density"])))
    exact_viol_match = bool((re_turns_df["hard_violations"] == orig_turns["hard_violations"]).all())

    assert max_psem_err < 1e-6 and max_mattr_err < 1e-6 and max_dens_err < 1e-6 and exact_viol_match
    print(f"  12,000-Turn CSV Comparison : EXACT MATCH (max |Δp_sem|={max_psem_err:.2e}, max |Δmattr|={max_mattr_err:.2e}, violations_match={exact_viol_match})")
else:
    print("  12,000-Turn Recomputation  : Completed all 12,000 turns from scratch (attach results-kv-vs-spa to cross-diff CSVs)")
    print(re_traj_df.groupby("arm_id")[["mean_p_sem", "mean_mattr", "repetition_collapse_flag", "total_hard_violations"]].agg({"mean_p_sem": "mean", "mean_mattr": "mean", "repetition_collapse_flag": lambda s: np.mean(s)*100, "total_hard_violations": "sum"}).round(4))

# ==============================================================================
# STAGE 4: COMPLETE STATISTICAL SUITE (PAIRED T, WILCOXON, WELCH, BH-FDR, & LMM)
# ==============================================================================
print("\n" + "=" * 88)
print("STAGE 4: FULL STATISTICAL REPRODUCTION (TABLE 3 CONTRASTS & TABLE 4 MIXEDLM)")
print("=" * 88)

ref_s = re_traj_df[re_traj_df["arm_id"] == "A2"].set_index("cell_id")["mean_p_sem"]
raw_p_paired, raw_p_welch, rows_t3 = [], [], []

for arm in ["A1", "A3", "A4", "A5", "A6"]:
    tst_s = re_traj_df[re_traj_df["arm_id"] == arm].set_index("cell_id")["mean_p_sem"].loc[ref_s.index]
    diffs = tst_s.values - ref_s.values
    t_rel, p_rel = stats.ttest_rel(tst_s.values, ref_s.values)
    w_stat, p_wil = stats.wilcoxon(tst_s.values, ref_s.values)
    t_wel, p_wel = stats.ttest_ind(tst_s.values, ref_s.values, equal_var=False)
    dz = float(np.mean(diffs) / np.std(diffs, ddof=1))
    raw_p_paired.append(float(p_rel))
    raw_p_welch.append(float(p_wel))
    rows_t3.append({"arm": arm, "mean_diff": round(float(np.mean(diffs)), 4), "paired_dz": round(dz, 3), "t_paired": round(float(t_rel), 3), "p_wilcoxon": round(float(p_wil), 5)})

_, fdr_p = compute_benjamini_hochberg_fdr(raw_p_paired)
_, fdr_w = compute_benjamini_hochberg_fdr(raw_p_welch)
for i, r in enumerate(rows_t3):
    r["p_paired_fdr"] = round(float(fdr_p[i]), 5)
    r["p_welch_fdr"] = round(float(fdr_w[i]), 5)
print("  Table 3 Recomputed Contrasts:\n", pd.DataFrame(rows_t3).to_string(index=False))

# Linear Mixed-Effects Model (Table 4)
re_turns_lmm = re_turns_df.merge(re_traj_df[["trajectory_id", "arm_id", "cell_id"]], on="trajectory_id")
re_turns_lmm["turn_centered"] = (re_turns_lmm["turn_index"] - 25.5) / 50.0
lmm_fit = smf.mixedlm("p_sem ~ C(arm_id, Treatment(reference='A2')) * turn_centered", data=re_turns_lmm, groups=re_turns_lmm["cell_id"]).fit(reml=True)
print("\n  Table 4 Recomputed LMM Fixed Effects:\n", lmm_fit.summary().tables[1])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


STAGE 1: BYTE-LEVEL DATA INTEGRITY & EXECUTION COMPLETENESS
  Archive Byte SHA-256 : 68caad94812bcf582ce7ff40c9d0ecadc6f13415df380684f43fa8ad7f91bd14
  240-JSON Content SHA : 863aea23665adba314fb6f7b429000cb938f49a9ae1576904c31616935052bdb
  Factorial Grid       : 240/240 unique (arm, model, persona) trajectories x 50 turns = 12,000 turns
  Empty Response Audit : 23/12,000 turns (0.19%) across 11 trajectories
  Boundary Overlap     : 5 trajectories had empty turns inside T1-5 or T46-50 MATTR windows

STAGE 2: ARM IMPLEMENTATION & MECHANISM VERIFICATION (A1-A6)
  Module baselines (A1/A2/A3)       | SHA256[:12]=99cc2662d7db | Classes=['TurnExecutionTelemetry', 'TrajectoryRecord'] | Funcs=['build_turn_messages', 'execute_single_turn_generation', 'run_baseline_trajectory', 'to_json']
  Module belitsky_steering (A4)     | SHA256[:12]=5dfe93067bd8 | Classes=['ContrastivePromptPair', 'BelitskySteeringConfig', 'PersonaSteeringVectors'] | Funcs=['resolve_target_layers', 'extract_contrastive_ste

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  12,000-Turn CSV Comparison : EXACT MATCH (max |Δp_sem|=5.55e-17, max |Δmattr|=1.11e-16, violations_match=True)

STAGE 4: FULL STATISTICAL REPRODUCTION (TABLE 3 CONTRASTS & TABLE 4 MIXEDLM)
  Table 3 Recomputed Contrasts:
 arm  mean_diff  paired_dz  t_paired  p_wilcoxon  p_paired_fdr  p_welch_fdr
 A1    -0.0360     -0.958    -6.058     0.00000       0.00000      0.03333
 A3    -0.0192     -0.775    -4.903     0.00003       0.00002      0.19387
 A4    -0.0345     -0.965    -6.101     0.00000       0.00000      0.03333
 A5    -0.0118     -0.210    -1.329     0.05805       0.19151      0.44240
 A6    -0.0361     -1.008    -6.375     0.00000       0.00000      0.03333

  Table 4 Recomputed LMM Fixed Effects:
                                                      Coef. Std.Err.        z  \
Intercept                                            0.478    0.010   46.754   
C(arm_id, Treatment(reference='A2'))[T.A1]          -0.036    0.002  -21.278   
C(arm_id, Treatment(reference='A2'))[T.A3]  

/tmp/ipykernel_25/134178631.py:234: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  lmm_fit = smf.mixedlm("p_sem ~ C(arm_id, Treatment(reference='A2')) * turn_centered", data=re_turns_lmm, groups=re_turns_lmm["cell_id"]).fit(reml=True)
